## Information theory


### Information Content (Surprise)

**When something unlikely happens, it carries the more information.**

`I(x) = -log(p(x))`

**The less likely it happens, the more information it carries**

In [2]:
import pandas
import math

df = pandas.DataFrame({
    "Event": ["Fair coin heads", "Rolling a 6", "1-in-1000 event", "Certain event"],
    "Probability": [0.5, 1/6, 1/1000, 1],
    "Information": [
        -math.log2(0.5),
        -math.log2(1/6),
        -math.log2(1/1000),
        -math.log2(1)
    ]
})
df

,Event,Probability,Information
0,Fair coin heads,0.500000,1.000000
1,Rolling a 6,0.166667,2.584963
2,1-in-1000 event,0.001000,9.965784
3,Certain event,1.000000,-0.000000


## Entropy (Average Suprise)

**Entropy is the expected surprise across all possible outcomes of a distribution**

`H(P) = -sum(p(x) * log(p(x))) for all x`

**Entropy measures the irreducible uncertainty in a distribution**
* A fair coin heads(50% up, 50% down) carries more uncertainty than a biased coin (99% up, 1% down)

In [5]:
df = pandas.DataFrame({
    "Event": ["Fair coin heads", "Biased coin heads"],
    "Probability": [0.5, 0.99],
    "Entropy": [
        - sum(p * math.log2(p) for p in [0.5, 0.5]),
        - sum(p * math.log2(p) for p in [0.99, 0.01])
    ]
})
df

,Event,Probability,Entropy
0,Fair coin heads,0.50,1.000000
1,Biased coin heads,0.99,0.080793


## Cross Entropy

**The average surprise then you use distribution Q to encode events that actually come from distribution P**

`H(P, Q) = -sum(p(x) * log(q(x))) for all x`

P is the true distribution(eg. the labels). Q is your model's predictions.
If Q matches P perfectly, cross-entroy equals entropy.
**Any mismatch makes it larger**

## KL Divergence (Distance between distributions)

**Measures how much extra surprise you get from using Q instead of P**

`D_KL(P || Q) = sum(p(x) * log(p(x) / q(x))) for all x = H(P, Q) - H(P)`

**So corss-entropy is the entropy plus KL divergence. Since entropy of the true distribution is constant during training, minimizing cross-entropy is hte same as minimizing KL divergence.**

**Which means you are pushing your model's distribution toward the ture distribution**

## Mutual Information

**Measures how much knowing one variable tells you about another**

`I(X; Y) = H(X) - H(X|Y) = H(X) + H(Y) - H(X, Y)`

## Conditional Entropy

**Measures how much uncertainty remains about Y after you observe X**

`H(Y|X) = H(X,Y) - H(X)`

Examples:
* Knowing the celcius degrees eliminates all uncertainty of fahrenheit degrees.
* Knowing the coin's face towards doesn't effect the uncertainty of today's weather.

`0 <= H(Y|X) <= H(Y)`

In [6]:
df = pandas.DataFrame({
    "Event": ["X: Tempreature celcius, Y: Tempreature fahrenheit",
        "X: Weather, Y: Coin"],
    "H(Y|X)": [
        "0", "H(Y)"
    ]
})
df

,Event,H(Y|X)
0,"X: Tempreature celcius, Y: Tempreature fahrenheit",0
1,"X: Weather, Y: Coin",H(Y)


## Joint Entropy

**Entropy of the joint distribution of X and Y together**

`H(X,Y) = sum sum p(x, y) * log(p(x, y)) for all x, y`

`H(X, Y) <= H(X) + H(Y)`

## Build your own

In [7]:
import math

def information_content(p, base=2):
    if p <= 0 or p > 1:
        return float('inf') if p <= 0 else 0.0
    return -math.log(p) / math.log(base)

def entropy(probs, base=2):
    return sum(
        p * information_content(p, base) for p in probs if p > 0
    )

fair_coin = [0.5, 0.5]
biased_coin = [0.99, 0.01]
fair_die = [1/6] * 6

print(entropy(fair_coin))
print(entropy(biased_coin))
print(entropy(fair_die))



1.0
0.08079313589591118
2.584962500721156


In [8]:
def cross_entropy(p, q, base=2):
    total = 0.0
    for pi, qi in zip(p, q):
        if pi > 0:
            if qi <= 0:
                return float('inf')
            total += pi * (-math.log(qi) / math.log(base))
    return total

def kl_divergence(p, q, base=2):
    return cross_entropy(p, q, base) - entropy(p, base)

true_dis = [0.7, 0.2, 0.1]
good_model = [0.6, 0.25, 0.15]
bad_model = [0.1, 0.1, 0.8]

print(f"Entropy of true distribution: {entropy(true_dis)}")
print(f"Good model, cross_entropy: {cross_entropy(true_dis, good_model):.4f}, kl_divergence: {kl_divergence(true_dis, good_model):.4f}")
print(f"Bad model, cross_entropy: {cross_entropy(true_dis, bad_model):.4f}, kl_divergence: {kl_divergence(true_dis, bad_model):.4f}")



Entropy of true distribution: 1.1567796494470395
Good model, cross_entropy: 1.1896, kl_divergence: 0.0328
Bad model, cross_entropy: 3.0219, kl_divergence: 1.8651


In [16]:
def softmax(logits):
    max_logit = max(logits)
    exps = [math.exp(z - max_logit) for z in logits]
    total = sum(exps)
    return [e / total for e in exps]

def cross_entropy_loss(true_class, logits):
    probs = softmax(logits)
    return -math.log(probs[true_class])

logits = [10.0, 10.0, 0.1]
true_class = 0

probs = softmax(logits)
print(probs)
print(f"Probability of true class: {probs[true_class]:.4f}")

loss = cross_entropy_loss(true_class, logits)
print(f"Cross entropy loss: {loss:.4f}")


[0.4999874566441654, 0.4999874566441654, 2.508671166919672e-05]
Probability of true class: 0.5000
Cross entropy loss: 0.6932
